In [ ]:
from sklearn.datasets import load_breast_cancer
import numpy as np

In [ ]:
def explorer_dataset(dataset, filter=None):
    X, y = dataset.data, dataset.target
    if filter is not None:
        class_index = y == filter
        X = X[class_index]
        y = y[class_index]
    print(f"Lignes, colonnes : {X.shape}")
    classes, counts = np.unique(y, return_counts=True)
    total = len(y)
    for class_index, count in zip(classes, counts):
        percentage = count/total*100
        print(
            f"Class {class_index} "
            f"({dataset.target_names[class_index]}) : {count} cases "
            f"({percentage:.2f}%)"
        )

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier

In [ ]:
modeles = {
        "Logistic Regression": LogisticRegression(max_iter=5000),
        "Decision Tree": DecisionTreeClassifier(random_state=42),
        "KNN": KNeighborsClassifier(n_neighbors=3)
    }
print("------Breast Cancer Dataset-------")
breast_cancer_dataset = load_breast_cancer()
explorer_dataset(breast_cancer_dataset)
explorer_dataset(breast_cancer_dataset, 0)
explorer_dataset(breast_cancer_dataset, 1)

In [ ]:
from sklearn.metrics import accuracy_score

In [ ]:
def entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test):
    modele.fit(X_train, y_train)
    y_pred = modele.predict(X_test)
    accuracy = accuracy_score(y_test, y_pred)
    return accuracy

In [ ]:
from sklearn.model_selection import train_test_split

In [ ]:
def arene(modeles, X_train, X_test, y_train, y_test):
    classement = []

    for nom, modele in modeles.items():
        score = entrainer_et_evaluer(modele, X_train, X_test, y_train, y_test)
        classement.append((nom, score))
    classement.sort(key=lambda x: x[1], reverse=True)

    print("\n🏆 Classement des modèles :\n")
    for i, (nom, score) in enumerate(classement, start=1):
        print(f"{i}. {nom} → {score:.1%}")
    return classement

In [ ]:
X, y = breast_cancer_dataset.data, breast_cancer_dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
classement_breast_cancer = arene(modeles, X_train, X_test, y_train, y_test)

In [ ]:
from sklearn.cluster import KMeans
import pandas as pd
from sklearn.metrics import adjusted_rand_score

In [ ]:
def clustering_aveugle(X):
    kmeans = KMeans(n_clusters=2, random_state=42, n_init=10)
    clusters = kmeans.fit_predict(X)
    return clusters

def comparer_clusters(y_class, clusters):
    table = pd.crosstab(y_class, clusters)
    print("\nClasses vs clusters :\n")
    print(table)
    score = adjusted_rand_score(y_class, clusters)
    print(f"Score de correspondance classes vs clusters: {score:.1%}")

In [ ]:
clusters = clustering_aveugle(X)
comparer_clusters(y, clusters)

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
def afficher_classement(resultats):
    noms = [nom for nom, score in resultats]
    scores = [score for nom, score in resultats]

    plt.figure(figsize=(8, 4))
    plt.bar(noms, scores)

    plt.title("Comparaison des modèles")
    plt.xlabel("Algorithme")
    plt.ylabel("Accuracy")
    plt.ylim(0, 1)

    plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [ ]:
def afficher_matrice_confusion(dataset_value, dataset_name, champion_modele_fct, champion_modele_name, 
                               X_train, X_test, y_train, y_test):
    champion_modele_fct.fit(X_train, y_train)
    y_pred = champion_modele_fct.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=dataset_value.target_names
    )
    disp.plot()
    plt.title(f"Matrice de confusion {dataset_name} - {champion_modele_name}")
    plt.show()

In [ ]:
print("------Breast Cancer Graphs-------")
afficher_classement(classement_breast_cancer)
afficher_matrice_confusion(breast_cancer_dataset, "Breast Cancer", LogisticRegression(max_iter=5000),
                           "Logistic Regression", X_train, X_test, y_train, y_test)

print("\n")

In [ ]:
from sklearn.preprocessing import StandardScaler

In [ ]:
def comparer_scaling(modeles, X_train, X_test, y_train, y_test):
    resultats = []
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    for nom, modele in modeles.items():
        score_brut = entrainer_et_evaluer(
            modele,
            X_train,
            X_test,
            y_train,
            y_test
        )
        score_scaled = entrainer_et_evaluer(
            modele,
            X_train_scaled,
            X_test_scaled,
            y_train,
            y_test
        )

        gain = score_scaled - score_brut

        resultats.append(
            (nom, score_brut, score_scaled, gain)
        )

    resultats.sort(key=lambda x: x[3], reverse=True)

    print("\n🏋️ Classement des gains du scaling\n")

    print(
        f"{'Algo':<25}"
        f"{'Brut':>10}"
        f"{'Scalé':>10}"
        f"{'Gain':>10}"
    )
    for nom, brut, scale, gain in resultats:
        print(
            f"{nom:<25}"
            f"{brut:>9.1%}"
            f"{scale:>10.1%}"
            f"{gain:+9.1%}"
        )
    return resultats

In [ ]:
from sklearn.base import clone

In [ ]:
def comparer_honnete_vs_triche(modele, X_train, X_test, y_train, y_test):
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
    score_honnete = entrainer_et_evaluer(
        clone(modele),
        X_train_scaled,
        X_test_scaled,
        y_train,
        y_test
    )
    score_triche = entrainer_et_evaluer(
        clone(modele),
        X_train,
        X_test,
        y_train,
        y_test
    )
    delta = score_triche - score_honnete
    print(f"Accuracy honnête : {score_honnete:.1%}")
    print(f"Accuracy triche  : {score_triche:.1%}")
    print(f"Mensonge  : {delta:+.1%}")
    return score_honnete, score_triche, delta

In [ ]:
print("Manche 1 : buff")
comparer_scaling(modeles, X_train, X_test, y_train, y_test)
print("\nManche 2 : triche")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
comparer_honnete_vs_triche(LogisticRegression(max_iter=5000),
                           X_train, X_test, y_train, y_test 
                           )

In [ ]:
from sklearn.datasets import load_wine

In [ ]:
print("\n")
print("------Wine Dataset-------")
wine_dataset = load_wine()
explorer_dataset(wine_dataset)
explorer_dataset(wine_dataset, 0)
explorer_dataset(wine_dataset, 1)
explorer_dataset(wine_dataset, 2)

X, y = wine_dataset.data, wine_dataset.target
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)
classement_wine = arene(modeles, X_train, X_test, y_train, y_test)

clusters = clustering_aveugle(X)
comparer_clusters(y, clusters)

print("------Wine Graphs-------")
afficher_classement(classement_wine)
afficher_matrice_confusion(wine_dataset, "Wine", LogisticRegression(max_iter=5000),
                           "Logistic Regression", X_train, X_test, y_train, y_test)
print("Manche 1 : buff")
comparer_scaling(modeles, X_train, X_test, y_train, y_test)
print("\nManche 2 : triche")
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42, stratify=y
)
comparer_honnete_vs_triche(LogisticRegression(max_iter=5000),
                           X_train, X_test, y_train, y_test 
                           )